# Capítulo 3 — Como medir se a IA acertou

## Missão
Construir a sua **caixa de ferramentas de avaliação**: métricas de linguagem (entropia, perplexidade), métricas exatas (acerto exato, F1, sobreposição de n-gramas), similaridade por embeddings, **IA como juiz** (e os vieses dela) e um ranking de modelos por comparação em pares. Sem avaliação, não dá para saber se uma mudança melhorou ou piorou o produto.

**No fim deste caderno você terá:** os módulos `meu_assistente/metricas.py` e `meu_assistente/juiz.py`, que os próximos capítulos usam para medir o assistente.

**Leitura essencial (opcional, livro em inglês):**
- Understanding Language Modeling Metrics · PDF p. 142
- Exact Evaluation · PDF p. 149
- AI as a Judge · PDF p. 160
- Ranking Models with Comparative Evaluation · PDF p. 172

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap03_como_medir_se_a_ia_acertou.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, re
sys.path.insert(0, "../comum")                 # ia.py (acesso aos modelos) e dados.py (dados da Nimbus)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_assistente/
os.makedirs("../meu_assistente", exist_ok=True)
open("../meu_assistente/__init__.py", "a").close()        # meu_assistente/ é o projeto que você constrói ao longo do curso
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
import numpy as np
import pandas as pd
from ia import *
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Modelo:", MODELO, "| modo simulado:", FAKE, "| seu pacote: meu_assistente")

## Ideia em 30 segundos
Avaliar modelos de base é difícil porque as respostas são abertas: há muitas formas certas de dizer a mesma coisa. Existem três famílias de métodos:
- **exatas**: o código confere (testes unitários, acerto exato, sobreposição com uma referência);
- **por similaridade**: embeddings dizem se o sentido é parecido;
- **subjetivas**: outro modelo dá a nota (IA como juiz). É barato e flexível, mas tem vieses que você precisa medir.

### 3.1 **Entropia.** Calcule, em bits, a entropia de três distribuições sobre 4 tokens: uniforme `[0.25]*4`, concentrada `[0.97, 0.01, 0.01, 0.01]` e intermediária `[0.5, 0.3, 0.15, 0.05]`. Quanto maior a entropia, mais "em dúvida" o modelo está.

### 3.2 **Perplexidade a partir dos logprobs.** Perplexidade = `exp(-média dos logprobs)`. Peça duas respostas com `logprobs=True`: uma factual ("Quanto é 12 x 12? Responda só o número.") e uma criativa ("Invente um nome para um gato."). Calcule a perplexidade de cada uma. Qual o modelo gerou com mais certeza?

### 3.3 **Correção funcional (o código passa nos testes?).** Peça ao modelo uma função Python `eh_palindromo(texto)` que ignore espaços, maiúsculas e acentos. Extraia o bloco de código da resposta (regex entre crases), execute com `exec` num dicionário separado e rode 4 testes. Se não vier código, conte como falha.

### 3.4 **Acerto exato e F1 de tokens.** Escreva `normalizar(t)` (minúsculas, sem acentos, sem pontuação), `exata(resposta, referencia)` (0 ou 1) e `f1_tokens(resposta, referencia)` (F1 entre os conjuntos de palavras, como no SQuAD). Teste com a referência "R$ 29 por usuário por mês." e as respostas "29 reais por usuário ao mês" e "O plano custa R$ 59.".

### 3.5 **Sobreposição de n-gramas (a ideia do BLEU/ROUGE).** Escreva `sobreposicao(resposta, referencia, n=2)`: a fração dos n-gramas da referência que aparecem na resposta. Mostre para n=1 e n=2 com as mesmas frases. Por que métricas assim penalizam respostas certas escritas de outro jeito?

### 3.6 **Similaridade semântica.** Com `embeddings` e `similaridade`, compare a referência com: uma paráfrase correta, uma resposta errada mas parecida e um texto sem relação. A ordem das notas faz sentido?

### 3.7 **Módulo de métricas.** Junte num arquivo `metricas.py` as funções `normalizar`, `exata`, `f1_tokens`, `sobreposicao` e `similaridade_semantica(resposta, referencia)` (cosseno dos embeddings).

_A célula grava `meu_assistente/metricas.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_assistente/metricas.py


### 3.8 **IA como juiz.** Escreva um prompt de juiz com uma **rubrica** de 1 a 5 (5 = correta e completa segundo a referência; 1 = errada ou inventada) que devolve JSON `{"nota": int, "motivo": str}`. Avalie 3 respostas para "Quanto custa o plano Equipe?": uma certa, uma incompleta e uma errada.

### 3.9 **Viés de posição.** Juízes tendem a preferir a resposta que aparece primeiro. Peça ao juiz para escolher a melhor entre A e B (JSON `{"vencedor": "A"|"B"|"empate"}`), depois **troque a ordem** e pergunte de novo. O veredito é consistente?

### 3.10 **Módulo do juiz.** Escreva `juiz.py` com `julgar(pergunta, resposta, referencia)` (devolve `nota` e `motivo`) e `comparar(pergunta, a, b)`, que pergunta nas duas ordens e só declara vencedor se as duas concordarem; senão devolve `empate`.

_A célula grava `meu_assistente/juiz.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_assistente/juiz.py


### 3.11 **Ranking por comparação (Elo).** Quatro modelos fictícios disputaram partidas em pares (os resultados estão abaixo, já prontos). Implemente a atualização Elo (`esperado = 1 / (1 + 10**((rb - ra) / 400))`, `ra += K * (resultado - esperado)`, K=32), comece todos em 1.000, percorra as partidas 3 vezes e mostre o ranking.

### 3.12 **Use os módulos.** Importe `f1_tokens` e `similaridade_semantica` de `meu_assistente.metricas` e `julgar` de `meu_assistente.juiz`. Avalie a resposta do seu `nucleo.responder` para "Qual o horário do suporte?" com as três medidas, contra a referência de `PERGUNTAS`.

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero testar com o modelo de verdade (fora do modo simulado):
